In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

In [34]:
feature_names = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate",
    "category"
]

df = pd.read_csv("../datasets/kdd99/kddcup.data_10_percent_corrected", names=feature_names)
df.head()


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,category
0,0,tcp,http,SF,181,5450,0,0,0,0,...,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,normal.
1,0,tcp,http,SF,239,486,0,0,0,0,...,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0,normal.
2,0,tcp,http,SF,235,1337,0,0,0,0,...,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.
3,0,tcp,http,SF,219,1337,0,0,0,0,...,39,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.
4,0,tcp,http,SF,217,2032,0,0,0,0,...,49,1.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0,normal.


In [58]:
df["category"].unique()

array([0, 1])

In [36]:
# convert all attacks into 1, and let normal be 0
df["category"] = (df["category"] != "normal.").astype(int)
df["category"].unique().tolist()

[0, 1]

In [48]:
def scale_dataset(dataframe, oversample=False):
    # Identify symbolic (object) columns
    symbolic_cols = dataframe.select_dtypes(include=["object"]).columns.tolist()

    # One-hot encode symbolic columns
    df_encoded = pd.get_dummies(dataframe, columns=symbolic_cols)

    # Separate features and label
    X = df_encoded.iloc[:, :-1].values
    y = df_encoded.iloc[:, -1].values

    # Scale features
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Oversample if needed
    if oversample:
        ros = RandomOverSampler()
        X, y = ros.fit_resample(X, y)

    # Combine X and y for output
    data = np.hstack((X, np.reshape(y, (-1, 1))))

    return data, X, y

In [49]:
train, valid, test = np.split(df.sample(frac=1), [int(0.6*len(df)), int(0.8*len(df))])

/home/hal/Desktop/tcc/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [50]:
train, X_train, y_train = scale_dataset(train, oversample=False)
valid, X_valid, y_valid = scale_dataset(valid, oversample=False)
test, X_test, y_test = scale_dataset(test, oversample=False)

In [51]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

In [55]:
knn_model = KNeighborsClassifier(n_neighbors=2)
knn_model.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=2)

In [56]:
y_pred = knn_model.predict(X_test)

In [57]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       1.00      1.00      1.00     98785
        True       0.95      0.95      0.95        20

    accuracy                           1.00     98805
   macro avg       0.97      0.97      0.97     98805
weighted avg       1.00      1.00      1.00     98805



In [62]:
print(y_pred[:10])
print(X_test[:10])

[False False False False False False False False False False]
[[-0.06689033 -0.00345233 -0.03147156 ... -0.00318136 -0.00551033
   0.5543533 ]
 [-0.06689033 -0.00345233 -0.03147156 ... -0.00318136 -0.00551033
   0.5543533 ]
 [-0.06689033 -0.00392001 -0.03147156 ... -0.00318136 -0.00551033
  -1.80390374]
 ...
 [-0.06689033 -0.00368436 -0.03147156 ... -0.00318136 -0.00551033
   0.5543533 ]
 [-0.06689033 -0.00368436 -0.03147156 ... -0.00318136 -0.00551033
   0.5543533 ]
 [-0.06689033 -0.00392001 -0.03147156 ... -0.00318136 -0.00551033
  -1.80390374]]
